In [ ]:
from pyspark.sql import functions as F



In [ ]:
df_patients = spark.read.csv("Files/patients.csv",header=True)

In [ ]:
display(df_patients)

In [ ]:

df_patients_transformed = (
    df_patients
    .withColumn("BIRTHDATE", F.to_date("BIRTHDATE"))
    .withColumn("DEATHDATE", F.to_date("DEATHDATE"))

    .withColumn(
        "AGE",
        F.floor(
            F.months_between(
                F.coalesce(F.col("DEATHDATE"), F.current_date()),
                F.col("BIRTHDATE")
            ) / 12
        ).cast("int")
    )

    .withColumn(
        "AGE_GROUP",
        F.when(F.col("AGE").isNull(), "Unknown")
         .when(F.col("AGE") < 18, "0-17")
         .when(F.col("AGE") < 35, "18-34")
         .when(F.col("AGE") < 50, "35-49")
         .when(F.col("AGE") < 65, "50-64")
         .otherwise("65+")
    )

    .withColumn(
        "DECEASED_FLAG",
        F.when(F.col("DEATHDATE").isNotNull(), 1)
         .otherwise(0)
    )

    .withColumn(
        "FULL_NAME",
        F.concat_ws(" ", F.trim("FIRST"), F.trim("LAST"))
    )

    .withColumn("LAT", F.col("LAT").cast("double"))
    .withColumn("LON", F.col("LON").cast("double"))
    .withColumn("ZIP", F.col("ZIP").cast("string"))
)


In [ ]:
display(df_patients_transformed)

In [ ]:
path=spark.conf.get('spark.executorEnv.HC_Goldpath')
display(path)

In [ ]:

(
    df_patients_transformed.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "Healthcare_DE_Project.HC_Gold.dbo.DIM_Patients"
    )
)
